# Session 0 — Google Colab setup

**Goal:** prepare the Colab runtime for the ECCB DGAT hands-on tutorial.

This notebook clones the tutorial repository, installs the lightweight Python package,
downloads the **10x Genomics CytAssist Tonsil** RNA/ADT pair, and verifies the committed
pretrained prediction artifact. Full DGAT training is **not** run here.


## 1. Clone or refresh the tutorial repository


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git"
REPO_DIR = Path("/content/ECCB-2026-Tutorial")

if Path("/content").exists():
    if (REPO_DIR / "hands-on_tutorial" / "src" / "dgat_tutorial").is_dir():
        print(f"Repository already present at {REPO_DIR}")
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR / "hands-on_tutorial")
else:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "dgat_tutorial").is_dir():
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError(
            "Not in Colab and could not find hands-on_tutorial/. "
            "Clone the repo and start Jupyter inside hands-on_tutorial/."
        )

DGAT_DIR = Path.cwd() / "external" / "DGAT"
if not (DGAT_DIR / "utils" / "Preprocessing.py").is_file():
    DGAT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/DGAT.git", str(DGAT_DIR)], check=True)

print("Working directory:", Path.cwd())
print("Official DGAT preprocessing:", DGAT_DIR / "utils" / "Preprocessing.py")


## 2. Install the lightweight tutorial environment


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown", "anndata", "scanpy",
     "scikit-learn", "seaborn", "matplotlib", "pandas", "numpy", "scipy"],
    check=True,
)

print("Python:", sys.version.split()[0])
from dgat_tutorial.checkpoints import tutorial_paths
print("OK:", tutorial_paths(Path.cwd()).root)


## 3. Download and verify the Tonsil assets


In [ ]:
from pathlib import Path
import subprocess

cmd = ["bash", "scripts/download_dgat_assets.sh", "--data-only", "--dataset", "Tonsil"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
subprocess.run(cmd + ["--check-only"], check=True)

pred = Path("data/raw/dgat_predictions.csv")
meta = Path("data/raw/dgat_predictions.metadata.json")
assert pred.is_file(), f"Missing committed predictions: {pred}"
assert meta.is_file(), f"Missing prediction sidecar: {meta}"
print(f"Predictions present: {pred} ({pred.stat().st_size:,} bytes)")


## 4. Final Colab checkpoint


In [ ]:
from pathlib import Path
import sys

tutorial_root = Path.cwd().resolve()
assert (tutorial_root / "src" / "dgat_tutorial").is_dir()
sys.path.insert(0, str(tutorial_root / "src"))
from dgat_tutorial.checkpoints import tutorial_paths
from dgat_tutorial.data import find_dgat_h5ad_pair

paths = tutorial_paths(tutorial_root)
pair = find_dgat_h5ad_pair(paths.raw_data)
print(f"Tutorial root: {paths.root}")
print(f"Tonsil pair: {pair}")
print("Colab setup complete. Keep this runtime and continue with Session 1 notebooks.")
